In [ ]:
import pandas as pd
import re
import numpy as np

In [2]:
upl_19 = pd.read_csv('upl_goals_2019_20.csv')
upl_20 = pd.read_csv('upl_goals_2020_21.csv')
upl_21 = pd.read_csv('upl_goals_2021_22.csv')
upl_22 = pd.read_csv('upl_goals_2022_23.csv')
upl_23 = pd.read_csv('upl_goals_2023_24.csv')
upl_24 = pd.read_csv('upl_goals_2024_25.csv')
upl_25 = pd.read_csv('upl_goals_2025_26.csv')

In [3]:
upl_goals_19_25 = pd.concat([upl_19, upl_20, upl_21, upl_22, upl_23, upl_24, upl_25], ignore_index=True)

In [4]:
df = upl_goals_19_25.copy()

In [5]:
df.shape

(3319, 32)

In [6]:
# Split home_team column where format is 'Team A Vs Team B'
df['home_team'] = df['home_team'].str.replace(r'\s+Vs\s+', '|', regex=True)
split_teams = df['home_team'].str.split('|', expand=True)
df['home_team'] = split_teams[0].str.strip()
df.loc[split_teams[1].notna(), 'away_team'] = split_teams[1].str.strip()

In [7]:
df['home_team'] = df['home_team'].str.replace(r'\s+VS\s+', '|', regex=True)
split_teams = df['home_team'].str.split('|', expand=True)
df['home_team'] = split_teams[0].str.strip()
df.loc[split_teams[1].notna(), 'away_team'] = split_teams[1].str.strip()

In [8]:
club_name_corrections = {'Vipers FC':'Vipers SC',
 'Bright Stars FC': 'Soltilo Bright Stars FC',
 'Mbarara City':'Mbarara City FC',
 'Police':'Police FC',
 'Ondupraka FC':'Onduparaka FC',
 'Sc Villa':'SC Villa',
 'Gaddafi FC':'Entebbe UPPC FC',
 'Tooro United':'Tooro United FC',
 'Tooro FC':'Tooro United FC',
 'Mbaara City FC':'Mbarara City FC',
 'Onduparak FC':'Onduparaka FC',
 'Ondupararka FC':'Onduparaka FC',
 'KCCA':'KCCA FC'}

df['home_team'] = df['home_team'].replace(club_name_corrections)
df['away_team'] = df['away_team'].replace(club_name_corrections)

In [9]:
df['home_team'] = df['home_team'].str.lower().str.strip()
df['away_team'] = df['away_team'].str.lower().str.strip()

In [10]:
uppercase_terms = {'fc','sc','kcca','ura','myda','updf','nec','bul','uppc'}

def normalize_team_name(name):
    if pd.isna(name):
        return name
    parts = name.split()
    normalized = []
    for p in parts:
        # strip non-alphanumeric for checking and output
        core = re.sub(r'[^A-Za-z0-9]', '', p)
        if not core:
            continue
        if core.lower() in uppercase_terms:
            normalized.append(core.upper())
        else:
            normalized.append(core.title())
    return " ".join(normalized)

df['home_team'] = df['home_team'].apply(normalize_team_name)
df['away_team'] = df['away_team'].apply(normalize_team_name)

In [11]:
df.loc[df['goal_minute'] == '247','goal_minute'] = '90' # placed 90 as placeholder until more info is found(busoga united vs bul fc md24 2023/04/21)


In [12]:
df.loc[df['goal_minute'] == '90+','goal_minute'] = '90+2' # placed 90+2 as placeholder until more info is found(kitara fc vs busoga united fc md22 2021/05/08)

In [13]:
df.loc[df['goal_minute'] == '757','goal_minute'] = '80' # placed 80 as placeholder until more info is found(onduparaka vs kyetume md13 2019-11-08)

In [14]:
df.loc[df['goal_minute'] == '30 (p)', 'goal_type'] = 'Penalty'

In [15]:
df.loc[df['goal_minute'] == '30 (p)','goal_minute'] = '30'

In [16]:
df['has_goal'] = df['goal_minute'].notna().astype(int)

In [17]:

def convert_minute(val):
    if pd.isna(val):
        return np.nan
    val = str(val)
    if '+' in val:
        base, added = val.split('+')
        return int(base) + int(added)
    else:
        return int(val)

df['goal_minute_num'] = df['goal_minute'].apply(convert_minute)
df['in_added_time'] = df['goal_minute'].astype(str).str.contains('+',regex=False).astype(int)


In [18]:
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')

In [19]:
del df['League']

In [20]:
df.duplicated().sum()

np.int64(0)

In [21]:
def team_code(name):
    return name.replace(' ', '')[:3].upper()

df['match_id'] = (
    'UPL' +
    df['Season'].str[-2:] + '/' +
    df['home_team'].apply(team_code) + '/' +
    df['away_team'].apply(team_code) + '/' +
    pd.to_datetime(df['Date']).dt.strftime('%d-%m')
)


In [22]:
df['round'] = np.where(df['Match Day'] <= 15, 'Round 1', 'Round 2')


In [28]:
df[(df['goal_minute_num'] >=45)  & (df['goal_minute_num'] <= 50)]['goal_minute']

17        48
18        49
38        45
51        46
94        45
        ... 
3265      45
3271      47
3280    45+3
3304      49
3306      49
Name: goal_minute, Length: 273, dtype: object

In [29]:
def match_half(minute):
    if pd.isna(minute):
        return np.nan
    elif minute <= 45:
        return 'First Half'
    else:
        return 'Second Half'
    

df['match_half'] = df['goal_minute_num'].apply(match_half)


In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3319 entries, 0 to 3318
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Date             3319 non-null   datetime64[ns]
 1   Time             3319 non-null   object        
 2   Season           3319 non-null   object        
 3   Match Day        3319 non-null   int64         
 4   home_team        3319 non-null   object        
 5   away_team        3319 non-null   object        
 6   goal_minute      3169 non-null   object        
 7   team_side        3169 non-null   object        
 8   player_name      3169 non-null   object        
 9   goal_type        3169 non-null   object        
 10  has_goal         3319 non-null   int64         
 11  goal_minute_num  3169 non-null   float64       
 12  in_added_time    3319 non-null   int64         
 13  match_id         3319 non-null   object        
 14  round            3319 non-null   object 

In [31]:
df.head()

,Date,Time,Season,Match Day,home_team,away_team,goal_minute,team_side,player_name,goal_type,has_goal,goal_minute_num,in_added_time,match_id,round,match_half
0,2019-11-05,4:30 pm,2019/20,12,Kyetume FC,Busoga United FC,10,home,Noel Nasasira,Regular,1,10.0,0,UPL20/KYE/BUS/05-11,Round 1,First Half
1,2019-11-05,4:30 pm,2019/20,12,Kyetume FC,Busoga United FC,80,away,Dan Ssewava,Regular,1,80.0,0,UPL20/KYE/BUS/05-11,Round 1,Second Half
2,2019-11-05,4:30 pm,2019/20,12,Kyetume FC,Busoga United FC,90+2,home,Jonathan Mugabi,Regular,1,92.0,1,UPL20/KYE/BUS/05-11,Round 1,Second Half
3,2019-10-09,8:00 pm,2019/20,8,SC Villa,Wakiso Giants FC,55,home,Amir Kakomo,Regular,1,55.0,0,UPL20/SCV/WAK/09-10,Round 1,Second Half
4,2019-10-09,8:00 pm,2019/20,8,SC Villa,Wakiso Giants FC,74,home,David Owori,Regular,1,74.0,0,UPL20/SCV/WAK/09-10,Round 1,Second Half


In [32]:
df.to_csv('upl_goals_2019_2025_cleaned.csv', index=False)